# Nemotron v7.7 — Training Notebook (designed to SURPASS 0.85 LB)

**Optimized for RTX 6000 Pro Blackwell sm_120 (96 GB VRAM)** with the `all_categorical_splits/` dataset.

## Why this beats the public 0.85 LB notebook

The 0.85 notebook gets its memory efficiency from 3 things. We adopt all 3, then add **5 more wins on top**.

### What 0.85 does (we now do too)
| Their trick | Our v77 implementation | Memory saved |
|---|---|---|
| Mamba CUDA fast path (`is_fast_path_available=True` + Blackwell wheels) | Cell 1 installs `causal_conv1d` + `mamba_ssm` wheels; Cell 10 verifies kernel and enables | **~30 GB** |
| Cut Cross-Entropy loss (no logits materialization) | Cell 12 patches `model.backbone.forward` with `linear_cross_entropy` | **~17 GB** |
| Plain `torch.optim.AdamW` (Blackwell-stable) | Cell 12 overrides `create_optimizer` (β=0.9, 0.95) | Stable, +2 GB |
| MoE-tied LoRA across 128 experts | Cell 11 detects expert LoRA params, ties via grad-summing callback | Quality+memory |
| fp32 LoRA + bf16 base + fp32 router | Cell 11 explicit cast pass with verification | Stability |

### What WE add (their notebook doesn't have)
| Our edge | Why it matters |
|---|---|
| **Stratified per-category batching** | Each effective batch is one category → coherent gradient signal per category |
| **2:1 LoRA α/r ratio (64/32)** | Sharper updates than their 1:1 (32/32) — more aggressive adaptation |
| **NaN auto-halt callback** | They burn hours on poisoned model; we stop within 2 logging steps |
| **Per-epoch zipped checkpoints** | Direct submission-ready zips at each epoch boundary |
| **Step-level checkpoints + resume** | Cross-session continuation if Kaggle 12-hr session ends |
| **MAX_SEQ_LEN=6144** (vs their 8192) | Less padding waste; covers 99% of CoT in our dataset |

## Memory budget (bf16 + fast path + CCE, batch=2, seq=6144)
- 30B base in bf16: **~60 GB**
- Activations w/ GC + CCE: **~6 GB** (vs ~25 GB without fast path + CCE)
- Mamba fast-path workspace: **~3 GB** (vs ~30 GB without)
- LoRA (~50-100M params, fp32): **~0.4 GB weights + ~1 GB AdamW state**
- **Peak: ~70-75 GB** → 20+ GB headroom on 96 GB Blackwell ✓

## Eval-Server Contract Preserved
- `r = 32`, `alpha = 64`, `lora_dropout = 0`, plain LoRA (no DoRA/rsLoRA)
- vLLM applies `delta = (alpha/r) * B*A` — train math = inference math.
- `lm_head` NOT adapted (clean inference).

## Failure modes vs v76
| v76 problem | v77 fix |
|---|---|
| Pure-Python Mamba scan → 30 GB OOM | CUDA fast path → 3 GB |
| Logits tensor `[B, T, 131k]` → 17 GB | CCE — never allocates |
| `paged_adamw_8bit` → CUDA crash @ 3.6 hr (UVM bug) | `torch.optim.AdamW` |
| `int8` mode → 27 sec/step | `bf16` native → ~7 sec/step |
| No checkpoints | Per-epoch zips + step ckpts + resume |
| Silent NaN poisoning possible | NaN guard halts within 2 logs |


In [1]:
# ============================================================
# 1. INSTALL DEPENDENCIES
# ============================================================
# Strategy: do NOT reinstall transformers/torch/datasets — Kaggle ships
# newer versions paired with the GPU image. Only install the packages
# Kaggle is missing (trl, peft, cut-cross-entropy, bitsandbytes, wandb)
# and the Blackwell Mamba CUDA wheels.
#
# Use --no-deps so pip doesn't transitively pull an older transformers
# into our isolated dir. Append target_dir to sys.path (do NOT insert at
# 0) so Kaggle's bundled transformers/torch take precedence for any
# shared packages.

import subprocess, sys, os, glob
from pathlib import Path

TARGET_DIR  = "/kaggle/working/packages"
OFFLINE_DIR = "/kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages"
os.makedirs(TARGET_DIR, exist_ok=True)
if TARGET_DIR not in sys.path:
    sys.path.append(TARGET_DIR)  # APPEND, never insert(0) — Kaggle libs first


def _pip_install(pkgs, *, no_deps=True, no_index=False, find_links=None,
                 path_arg=None, label=None):
    """Install pkgs into TARGET_DIR. Returns True on success."""
    cmd = [sys.executable, "-m", "pip", "install", "-q",
           "--target", TARGET_DIR]
    if no_deps:
        cmd.append("--no-deps")
    if no_index:
        cmd.append("--no-index")
    if find_links:
        cmd += ["--find-links", find_links]
    if path_arg:
        cmd.append(path_arg)
    else:
        cmd += pkgs
    try:
        subprocess.check_call(cmd)
        if label:
            print(f"[ok] {label}")
        return True
    except Exception as e:
        if label:
            print(f"[warn] {label} failed: {e}")
        return False


def _find_wheel(pattern, search_paths):
    for base in search_paths:
        if os.path.isdir(base):
            for f in glob.glob(f"{base}/**/{pattern}", recursive=True):
                return f
    return None


# ---------- nvidia-cutlass (must come first, before any CUDA imports) ----------
CUTLASS_PATHS = [
    "/kaggle/input/datasets/rubyducklove/nvidia-cutlass",
]
cutlass_wheel = _find_wheel("nvidia_cutlass-*.whl", CUTLASS_PATHS) \
             or _find_wheel("cutlass-*.whl", CUTLASS_PATHS)
CUTLASS_AVAILABLE = bool(cutlass_wheel) and _pip_install(
    [], path_arg=cutlass_wheel, label=f"nvidia-cutlass <- {cutlass_wheel}"
)

# ---------- core deps Kaggle is missing ----------
# trl + peft + cut-cross-entropy: not in Kaggle's standard image
# bitsandbytes + wandb: also not standard
# All with --no-deps to keep Kaggle's transformers/torch authoritative.
PKG_LIST = ["trl", "peft", "datasets", "bitsandbytes", "wandb", "cut-cross-entropy"]

if os.path.isdir(OFFLINE_DIR):
    _pip_install(PKG_LIST, no_index=True, find_links=OFFLINE_DIR,
                 label=f"core deps (offline): {PKG_LIST}")
else:
    _pip_install(PKG_LIST, label=f"core deps (online): {PKG_LIST}")

# ---------- Blackwell Mamba CUDA wheels ----------
WHEEL_PATHS = [
    "/kaggle/input/datasets/mayukh18/nemotron-packages",
]
ccv_wheel = _find_wheel("causal_conv1d-*.whl", WHEEL_PATHS)
mssm_wheel = _find_wheel("mamba_ssm-*.whl", WHEEL_PATHS)

CAUSAL_CONV1D_AVAILABLE = bool(ccv_wheel) and _pip_install(
    [], path_arg=ccv_wheel, label=f"causal_conv1d <- {ccv_wheel}"
)
MAMBA_AVAILABLE = bool(mssm_wheel) and _pip_install(
    [], path_arg=mssm_wheel, label=f"mamba_ssm <- {mssm_wheel}"
)
FAST_PATH_AVAILABLE = MAMBA_AVAILABLE and CAUSAL_CONV1D_AVAILABLE

# Resolve any .pth files our installs dropped (e.g. namespace packages)
def _resolve_pth(d):
    for pth in Path(d).glob("*.pth"):
        with pth.open() as fp:
            rel = fp.read().strip()
            p = pth.parent / rel
            if p.exists():
                sys.path.append(str(p))

_resolve_pth(TARGET_DIR)

# ---------- Verify Kaggle's transformers is new enough ----------
import transformers
TRANSFORMERS_VERSION = tuple(int(x) for x in transformers.__version__.split(".")[:2])
TRANSFORMERS_PATH = transformers.__file__
NEW_ENOUGH = TRANSFORMERS_VERSION >= (4, 45)

# Wandb env var
os.environ["WANDB_MODE"] = "offline"

# ---------- Component availability flags (read by later cells) ----------
BNB_AVAILABLE = True   # we just installed it
WANDB_AVAILABLE = True
CCE_AVAILABLE = True

# Verify each is actually importable
for pkg, var in [("bitsandbytes", "BNB_AVAILABLE"),
                 ("wandb",        "WANDB_AVAILABLE"),
                 ("cut_cross_entropy", "CCE_AVAILABLE")]:
    try:
        __import__(pkg)
    except Exception:
        globals()[var] = False

print("=" * 60)
print(f"  Dependency status")
print("=" * 60)
print(f"  transformers     : {transformers.__version__}  ({TRANSFORMERS_PATH})")
print(f"                     {'OK (>=4.45 has Nemotron-H helpers)' if NEW_ENOUGH else 'TOO OLD — Nemotron-H may fail to load'}")
print(f"  nvidia-cutlass   : {'YES' if CUTLASS_AVAILABLE else 'NO'}")
print(f"  causal_conv1d    : {'YES' if CAUSAL_CONV1D_AVAILABLE else 'NO'}")
print(f"  mamba_ssm        : {'YES' if MAMBA_AVAILABLE else 'NO'}")
print(f"  Mamba fast path  : {'ENABLED (~30GB savings)' if FAST_PATH_AVAILABLE else 'DISABLED'}")
print(f"  cut-cross-entropy: {'YES (~17GB savings)' if CCE_AVAILABLE else 'NO'}")
print(f"  bitsandbytes     : {'YES' if BNB_AVAILABLE else 'NO'}")
print(f"  wandb            : {'YES (offline mode)' if WANDB_AVAILABLE else 'NO'}")

assert NEW_ENOUGH, (
    f"Kaggle's transformers ({transformers.__version__}) is older than 4.45. "
    "Nemotron-H remote code requires is_flash_attn_greater_or_equal_2_10. "
    "Either update Kaggle's bundled transformers or pin a newer version."
)

# ---------- Purge Kaggle utility-script mamba_ssm (has Mamba3 -> needs cutlass DSL) ----------
# Kaggle auto-attaches /kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/
# to sys.path. Its bundled mamba_ssm imports cutlass python DSL on load and crashes.
# Our installed Blackwell wheel doesn't. Force ours to win.
_BAD_PATH_FRAGS = ("nvidia_utility_script", "nvidia-utility-script")
sys.path[:] = [p for p in sys.path if not any(b in p for b in _BAD_PATH_FRAGS)]
# Also drop any pre-imported utility-script modules so our wheel re-imports clean
for _m in list(sys.modules):
    _mfile = getattr(sys.modules[_m], "__file__", "") or ""
    if any(b in _mfile for b in _BAD_PATH_FRAGS):
        del sys.modules[_m]
# Move TARGET_DIR to the FRONT (transformers already imported from Kaggle path)
if TARGET_DIR in sys.path:
    sys.path.remove(TARGET_DIR)
sys.path.insert(0, TARGET_DIR)
print("[ok] purged kaggle utility-script paths; TARGET_DIR is now first on sys.path")


[ok] nvidia-cutlass <- /kaggle/input/datasets/rubyducklove/nvidia-cutlass/nvidia_cutlass-4.2.0.0-py3-none-any.whl
[ok] core deps (offline): ['trl', 'peft', 'datasets', 'bitsandbytes', 'wandb', 'cut-cross-entropy']
[ok] causal_conv1d <- /kaggle/input/datasets/mayukh18/nemotron-packages/causal_conv1d-1.6.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
[ok] mamba_ssm <- /kaggle/input/datasets/mayukh18/nemotron-packages/mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
  Dependency status
  transformers     : 5.0.0  (/usr/local/lib/python3.12/dist-packages/transformers/__init__.py)
                     OK (>=4.45 has Nemotron-H helpers)
  nvidia-cutlass   : YES
  causal_conv1d    : YES
  mamba_ssm        : YES
  Mamba fast path  : ENABLED (~30GB savings)
  cut-cross-entropy: YES (~17GB savings)
  bitsandbytes     : YES
  wandb            : YES (offline mode)
[ok] purged kaggle utility-script paths; TARGET_DIR is now first on sys.path


In [2]:
# ============================================================
# 2. IMPORTS & ENVIRONMENT
# ============================================================
# Disable transformers's torchvision integration before any transformers
# import. Kaggle's bundled torchvision was compiled against CUDA 12.8 but
# torch is on CUDA 13.0; if transformers tries to import torchvision (e.g.
# via beit/detr loss utils) we crash with a CUDA major-version mismatch.
# We don't use any vision features here, so disabling the flag is safe.

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import stat, shutil, zipfile, time, json, re, glob
import numpy as np
import torch
import torch.nn.functional as F
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, TrainerCallback, BitsAndBytesConfig,
)
from peft import (
    LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training,
)
from trl import SFTTrainer, SFTConfig
from tqdm.auto import tqdm

if WANDB_AVAILABLE:
    import wandb

print(f"PyTorch       : {torch.__version__}")
print(f"GPU           : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
print(f"VRAM          : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(f"transformers  : {__import__('transformers').__version__}")
print(f"bitsandbytes  : {'YES' if BNB_AVAILABLE else 'NO'}")
print(f"W&B           : {'offline' if WANDB_AVAILABLE else 'disabled'}")


[INFO] Running in WANDB offline mode
PyTorch       : 2.10.0+cu128
GPU           : NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM          : 95.0 GB
transformers  : 5.0.0
bitsandbytes  : YES
W&B           : offline


In [ ]:
# ============================================================
# 2b. WEIGHTS & BIASES — OFFLINE MODE
# ============================================================
WANDB_PROJECT  = "nemotron-v76"
WANDB_RUN_NAME = "v79-mega21k-a64-lr5e-5-1ep"
WANDB_DIR      = "/kaggle/working"

if WANDB_AVAILABLE:
    wandb.init(
        project=WANDB_PROJECT,
        name=WANDB_RUN_NAME,
        dir=WANDB_DIR,
        config={
            "model": "Nemotron-3-Nano-30B-A3B (NF4 4-bit QLoRA)",
            "data": "mega_train_21k (21,100 deduped records, RFT-overrides on hard cats)",
            "lora_rank": 32,
            "lora_alpha": 64,
            "learning_rate": 5e-5,
            "num_epochs": 1,
            "batch_size": 2,
            "grad_accum": 2,
            "effective_batch": 4,
            "max_seq_len": 8192,
            "lora_targets": "q,k,v,o,in,out,up,down (no lm_head, no embed_tokens)",
            "lora_dropout": 0.0,
            "warmup_steps": 100,
            "scheduler": "cosine",
            "packing": False,
            "stratified": True,
            "bf16_compute": True,
            "quantization": "NF4 (double_quant)",
            "optimizer": "paged_adamw_8bit",
            "adapter_kind": "plain LoRA on bf16 base + Mamba fast path + CCE",
        },
        tags=["nemotron", "lora", "v79", "mega21k", "rft_override"],
    )
    print(f"W&B offline run initialized: {wandb.run.dir}")
else:
    print("W&B not available — training metrics logged to stdout only.")

In [3]:
# ============================================================
# 3. TRITON FIXES — defensive (rmsnorm patch + ptxas-blackwell)
# ============================================================
# IMPORTANT: the rmsnorm patch loop must NOT use a bare `hasattr(mod, ...)`
# over all sys.modules. transformers/peft/trl use lazy module loaders that
# turn arbitrary `hasattr()` into a full submodule import, which on this
# Kaggle image triggers
#     transformers.models.beit.image_processing_pil_beit
#       -> from torchvision.transforms.v2 import functional
#       -> torchvision.extension._check_cuda_version()
#       -> CUDA mismatch crash (PyTorch=13.0, torchvision=12.8).
#
# The patch only ever needs to touch mamba_ssm / mamba-style modules, so we
# filter by name. We also skip the patch entirely when the Mamba CUDA fast
# path is enabled — its native kernel does rmsnorm correctly already.

def _pure_rmsnorm_fn(x, weight, bias=None, z=None, eps=1e-5,
                     group_size=None, norm_before_gate=True, upcast=True):
    dtype = x.dtype
    if upcast: x = x.float()
    var = x.pow(2).mean(-1, keepdim=True)
    y = x * torch.rsqrt(var + eps)
    out = y * weight.float()
    if bias is not None: out = out + bias.float()
    if z is not None:    out = out * F.silu(z.float())
    return out.to(dtype)


# --- rmsnorm patch (only when fast path OFF and only on mamba modules) ---
if not globals().get('FAST_PATH_AVAILABLE', False):
    patched = []
    for name, mod in list(sys.modules.items()):
        # Filter strictly: only mamba/ssm modules. Touching transformers/peft/trl
        # via hasattr() triggers their lazy loaders -> torchvision crash.
        nlow = name.lower()
        if not any(k in nlow for k in ("mamba", "ssm", "selective_scan", "rmsnorm")):
            continue
        # Skip transformers's own mamba implementations (lazy-loaded too)
        if nlow.startswith("transformers"):
            continue
        try:
            if hasattr(mod, "rmsnorm_fn"):
                mod.rmsnorm_fn = _pure_rmsnorm_fn
                patched.append(name)
        except Exception:
            pass
    if patched:
        print(f"[ok] rmsnorm_fn patched in: {patched}")
    else:
        print("[info] no mamba module had rmsnorm_fn to patch (will be patched after model load)")
else:
    print("[info] rmsnorm patch skipped — Mamba CUDA fast path is available, native kernel will be used")


# --- ptxas-blackwell shim (Triton -> Blackwell sm_120) ---
candidates = (
    glob.glob("/kaggle/usr/lib/notebooks/**/ptxas-blackwell", recursive=True)
    + glob.glob("/kaggle/usr/lib/notebooks/**/ptxas", recursive=True)
    + glob.glob("/usr/local/cuda*/bin/ptxas", recursive=True)
    + glob.glob("/usr/local/lib/python*/dist-packages/nvidia/cuda_nvcc/bin/ptxas",
                recursive=True)
)
src = next((c for c in candidates if "blackwell" in c), None) \
   or (candidates[0] if candidates else None)

if src and os.path.exists(src):
    dst = "/tmp/ptxas-blackwell"
    shutil.copy2(src, dst)
    os.chmod(dst, os.stat(dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    for v in ("TRITON_PTXAS_PATH", "TRITON_PTXAS_BLACKWELL_PATH",
              "TRITON_PTXAS_BIN", "TRITON_PTXAS"):
        os.environ[v] = dst
    try:
        import triton.backends.nvidia.compiler as nv_compiler
        try: nv_compiler.get_ptxas_version.cache_clear()
        except AttributeError: pass
        nv_compiler.get_ptxas_version = lambda arch: "release 12.8"
        from triton import knobs as triton_knobs
        for attr in ("ptxas", "ptxas_blackwell"):
            triton_knobs.nvidia.__dict__.pop(attr, None)
    except Exception as e:
        print(f"[warn] Triton cache clear: {e}")
    print(f"[ok] ptxas binary -> {dst}  (copied from {src})")
else:
    print("[warn] no ptxas binary found — Mamba Triton kernel may crash")


[info] rmsnorm patch skipped — Mamba CUDA fast path is available, native kernel will be used
[ok] ptxas binary -> /tmp/ptxas-blackwell  (copied from /usr/local/cuda-12.8/bin/ptxas)


In [ ]:
# ============================================================
# 4. HYPERPARAMETERS — v7.9 mega-21k (deduped union + RFT overrides)
# ============================================================
# Lessons from prior runs:
#   - 0.86 (proven) : v11_verified data (10,545 records) + alpha=64, LR=5e-5, 1 epoch
#   - 0.82 (regress): RFT-only mixed (7302 records) + alpha=64, LR=2e-4, 2 epochs
#                     → too small (lost diversity), too aggressive LR, too many epochs.
#
# THIS run: combine ALL training data versions (v6→v11) deduped by user prompt,
# then OVERRIDE hard-category CoTs with RFT-validated ones. Result: 21,100 records.
#
# Per-category breakdown (mega_train_21k/):
#   bit_manipulation         :  3609   (was 281 in RFT-only — 13× more diversity)
#   cipher                   :  7542   (was 1576 — 5× more, recovered v6/v7 unique)
#   cryptarithm_deduce       :   714   (RFT-validated where overlap exists)
#   cryptarithm_guess        :   180
#   equation_numeric_deduce  :   596
#   equation_numeric_guess   :   176
#   gravity                  :  2774
#   numeral                  :  2769
#   unit_conversion          :  2740
#   TOTAL                    : 21,100
#
# Hyperparameters: REVERT to proven 0.86 config (alpha=64, LR=5e-5, 1 epoch).

LORA_RANK           = 32          # eval-server cap (matches 0.85)
LORA_ALPHA          = 64          # proven 0.86 LB run (alpha=48 only got 0.85)
LORA_DROPOUT        = 0.0         # eval-server contract (vLLM math)

MAX_SEQ_LEN         = 8192        # tail-truncation in cell 9 handles long RFT CoTs
NUM_EPOCHS          = 1           # was 2 — RFT data is cleaner, 1 epoch suffices
BATCH_SIZE          = 2           # fast path saves ~30GB → bs=2 fits at seq 8192
GRAD_ACCUM          = 2           # eff batch = 4
LR                  = 5e-5        # PROVEN 0.86 LR (2e-4 caused overfit on smaller RFT-only run)
WARMUP_STEPS        = 100         # proven from 0.86 run
SAVE_EVERY_N_EPOCHS = 1
SAVE_EVERY_N_STEPS  = 200         # ~30 min recovery point

USE_PACKING             = False   # disabled — packing at seq 8192 is memory-heavy and mixes categories
USE_STRATIFIED_BATCHING = False   # incompatible with packing; packing wins for speed
USE_CCE                 = True    # Cut Cross-Entropy (saves ~17 GB)
USE_MAMBA_FAST_PATH     = True    # Fused CUDA scan (saves ~30 GB)

# MoE LoRA strategy:
#   "exclude"  → don't adapt MoE experts (v76 default — small but safe)
#   "tied"     → adapt MoE with weight tying (0.85's trick — we ADD this)
#   "full"     → adapt all 2967 experts independently (huge OOM risk)
MOE_LORA_MODE = "tied"

# Mode selector — bf16 native is fastest on Blackwell tensor cores
FORCE_MODE = "bf16"

if FORCE_MODE:
    MODE = FORCE_MODE
else:
    MODE = "bf16"

if MODE == "nf4":
    LR = 2e-4
    print("NF4 mode: LR=2e-4")
else:
    print(f"bf16 mode (FAST native tensor cores on Blackwell): LR={LR:.1e}")

USE_QLORA = MODE in ("nf4", "int8")

# Disable fast path if wheels not present (Cell 1 set FAST_PATH_AVAILABLE)
if USE_MAMBA_FAST_PATH and not FAST_PATH_AVAILABLE:
    print("[warn] USE_MAMBA_FAST_PATH=True but wheels missing — forcing OFF")
    USE_MAMBA_FAST_PATH = False
    print("       This will cost ~30 GB. Reducing BATCH_SIZE 2 -> 1, GRAD_ACCUM 2 -> 4")
    BATCH_SIZE  = 1
    GRAD_ACCUM  = 4

# Disable CCE if not installed
if USE_CCE and not CCE_AVAILABLE:
    print("[warn] USE_CCE=True but cut_cross_entropy missing — forcing OFF")
    USE_CCE = False

MODEL_PATH  = "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1"
OUTPUT_DIR  = "/kaggle/working/adapter"
CKPT_DIR    = "/kaggle/working/checkpoints"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CKPT_DIR,   exist_ok=True)

CATEGORY_FILES = [
    "train_cot_bit_manipulation.jsonl",
    "train_cot_cipher.jsonl",
    "train_cot_cryptarithm_deduce.jsonl",
    "train_cot_cryptarithm_guess.jsonl",
    "train_cot_equation_numeric_deduce.jsonl",
    "train_cot_equation_numeric_guess.jsonl",
    "train_cot_gravity.jsonl",
    "train_cot_numeral.jsonl",
    "train_cot_unit_conversion.jsonl",
]

# DATA_DIR_CANDIDATES — Option B style: NEW mixed (RFT+original) directory
# is searched FIRST. The original raw directory is kept as a fallback so the
# notebook still runs if you forget to attach the mixed dataset.
DATA_DIR_CANDIDATES = [
    # ── PRIMARY: mega-21k dataset (deduped v6+v7+v8+v9+v10+v11 union + RFT overrides) ──
    # Local (Mac dev):
    str(Path.cwd().parent / "data" / "processed" / "mega_train_21k"),
    str(Path.cwd() / "data" / "processed" / "mega_train_21k"),
    # Kaggle dataset paths (upload nemotron-mega-21k as a Kaggle dataset — the
    # notebook will find it under any of these path conventions):
    "/kaggle/input/datasets/manish756/nemotron-mega-21k",
    "/kaggle/input/manish-nemotron-mega-21k",
    "/kaggle/input/nemotron-mega-21k",
    # If you put it inside an all_categorical_splits/ subfolder when uploading:
    "/kaggle/input/datasets/manish756/nemotron-mega-21k/all_categorical_splits",
    "/kaggle/input/nemotron-mega-21k/all_categorical_splits",

    # ── FALLBACK: previous datasets ──────────────────────────────────────
    str(Path.cwd().parent / "data" / "processed" / "mixed_of_rft_original_data" / "all_categorical_splits"),
    str(Path.cwd() / "data" / "processed" / "mixed_of_rft_original_data" / "all_categorical_splits"),
    "/kaggle/input/datasets/asharamkanderiwal/nvidia-dataset/all_categorical_splits",
    "/kaggle/input/datasets/manish756/nemotron-dataset/all_categorical_splits",
    str(Path.cwd().parent / "data" / "processed" / "all_categorical_splits"),
    str(Path.cwd() / "data" / "processed" / "all_categorical_splits"),
]

assert LORA_RANK   <= 32,   f"LORA_RANK={LORA_RANK} exceeds eval cap 32"
assert MAX_SEQ_LEN <= 8192, f"MAX_SEQ_LEN={MAX_SEQ_LEN} exceeds eval max_model_len 8192"

print("=" * 60)
print("  v7.9 MEGA-21K — proven 0.86 hyperparams on 21K deduped+RFT data")
print("=" * 60)
print(f"  Mode             : {MODE}")
print(f"  Epochs           : {NUM_EPOCHS}  (epoch ckpt every {SAVE_EVERY_N_EPOCHS}, step ckpt every {SAVE_EVERY_N_STEPS})")
print(f"  LR               : {LR:.1e}  (warmup {WARMUP_STEPS}, cosine)")
print(f"  Batch            : {BATCH_SIZE}×{GRAD_ACCUM} = {BATCH_SIZE*GRAD_ACCUM} eff")
print(f"  LoRA             : r={LORA_RANK}, α={LORA_ALPHA}  (proven 0.86 — α=48 was only 0.85)")
print(f"  Max seqlen       : {MAX_SEQ_LEN}  (tail-truncation handles long RFT CoTs)")
print(f"  Stratified       : {USE_STRATIFIED_BATCHING}")
print(f"  Mamba fast path  : {USE_MAMBA_FAST_PATH}  (~30 GB saved)")
print(f"  Cut Cross-Entropy: {USE_CCE}              (~17 GB saved)")
print(f"  MoE LoRA mode    : {MOE_LORA_MODE}")
print(f"  Optimizer        : torch.optim.AdamW (Blackwell-stable)")
print(f"  Ckpt dir         : {CKPT_DIR}")
print(f"  Data sources     : (will pick first that exists)")
for d in DATA_DIR_CANDIDATES:
    marker = "✓" if os.path.isdir(d) else " "
    print(f"      [{marker}] {d}")


In [5]:
# ============================================================
# 5. CALLBACKS — progress + per-epoch ckpt zip + NaN auto-halt
# ============================================================
import math as _math

class LiveProgressCallback(TrainerCallback):
    def __init__(self):
        self.pbar = None
        self.start_time = None
    def on_train_begin(self, args, state, control, **kwargs):
        self.pbar = tqdm(total=state.max_steps, desc="Training",
                         unit="step", dynamic_ncols=True, file=sys.stdout)
        self.start_time = time.time()
    def on_step_end(self, args, state, control, **kwargs):
        if self.pbar is None:
            return
        elapsed = time.time() - self.start_time
        step    = state.global_step
        eta     = (elapsed / step) * (state.max_steps - step) if step > 0 else 0
        loss_str = (f"loss={state.log_history[-1]['loss']:.4f}"
                    if state.log_history and "loss" in state.log_history[-1] else "loss=...")
        self.pbar.set_postfix_str(f"{loss_str}  elapsed={elapsed/60:.1f}m  eta={eta/60:.1f}m")
        self.pbar.update(1)
        sys.stdout.flush()
    def on_train_end(self, args, state, control, **kwargs):
        if self.pbar:
            self.pbar.close()


class NaNGuardCallback(TrainerCallback):
    """Halt training if loss becomes NaN/Inf — saves hours of wasted compute.

    OUR EDGE: 0.85 has no NaN protection — if their training spikes to NaN,
    they keep burning compute on a poisoned model until the run completes.
    We detect within 1 logging step and stop cleanly, then the saved ckpt
    can be reloaded.
    """
    def __init__(self, max_consecutive=2):
        self.max_consecutive = max_consecutive
        self.bad_streak = 0
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None or "loss" not in logs:
            return
        loss = logs["loss"]
        if loss is None or _math.isnan(loss) or _math.isinf(loss):
            self.bad_streak += 1
            print(f"\n[NaN GUARD] Detected non-finite loss={loss} (streak={self.bad_streak})")
            if self.bad_streak >= self.max_consecutive:
                print(f"[NaN GUARD] HALTING training — model has diverged.")
                print(f"           Last good checkpoint should be in: {OUTPUT_DIR}/checkpoint-XXX")
                control.should_training_stop = True
        else:
            self.bad_streak = 0


class CheckpointZipCallback(TrainerCallback):
    """Saves a zipped LoRA adapter at the end of each epoch."""
    def __init__(self, ckpt_dir, output_dir, every_n=1):
        self.ckpt_dir   = ckpt_dir
        self.output_dir = output_dir
        self.every_n    = every_n
        self.epoch_losses = {}
    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        epoch = round(state.epoch)
        if epoch % self.every_n != 0:
            return
        epoch_dir = os.path.join(self.output_dir, f"epoch_{epoch:02d}")
        os.makedirs(epoch_dir, exist_ok=True)
        model.save_pretrained(epoch_dir)
        cfg_path = os.path.join(epoch_dir, "adapter_config.json")
        with open(cfg_path) as f:
            cfg = json.load(f)
        cfg["base_model_name_or_path"] = "metric/nemotron-3-nano-30b-a3b-bf16"
        with open(cfg_path, "w") as f:
            json.dump(cfg, f, indent=2)
        zip_name = f"adapter_epoch_{epoch:02d}.zip"
        zip_path = os.path.join(self.ckpt_dir, zip_name)
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
            for fname in sorted(os.listdir(epoch_dir)):
                fp = os.path.join(epoch_dir, fname)
                if os.path.isfile(fp):
                    zf.write(fp, arcname=fname)
        recent = [h["loss"] for h in state.log_history if "loss" in h]
        avg_loss = sum(recent[-10:]) / len(recent[-10:]) if recent else float("nan")
        self.epoch_losses[epoch] = avg_loss
        zip_mb = os.path.getsize(zip_path) / 1024 / 1024
        print(f"\n[Epoch {epoch:02d}] ✓ {zip_name}  ({zip_mb:.1f} MB)  avg_loss={avg_loss:.4f}")
        if WANDB_AVAILABLE and wandb.run is not None:
            wandb.log({"epoch_checkpoint/epoch": epoch,
                       "epoch_checkpoint/avg_loss": avg_loss,
                       "epoch_checkpoint/zip_mb": zip_mb}, step=state.global_step)
    def print_summary(self):
        if not self.epoch_losses:
            return
        print("\n  Checkpoint loss summary:")
        best = min(self.epoch_losses, key=self.epoch_losses.get)
        for ep, loss in sorted(self.epoch_losses.items()):
            mark = " ← best" if ep == best else ""
            print(f"    epoch {ep:02d}: loss={loss:.4f}{mark}")
        print(f"\n  Best checkpoint: adapter_epoch_{best:02d}.zip  (use for submission)")


class TiedMoEGradCallback(TrainerCallback):
    """Sums LoRA grads across MoE experts before optimizer.step().

    Required when MOE_LORA_MODE='tied' so all 128 expert slices receive
    the SAME gradient and remain identical after AdamW updates.
    """
    def on_pre_optimizer_step(self, args, state, control, **kwargs):
        try:
            _tie_grads()
        except NameError:
            pass  # _tie_grads not defined (Cell 11 not run yet, or MoE excluded)


ckpt_callback   = CheckpointZipCallback(CKPT_DIR, OUTPUT_DIR, SAVE_EVERY_N_EPOCHS)
nan_callback    = NaNGuardCallback(max_consecutive=2)
tied_callback   = TiedMoEGradCallback()
print("Callbacks ready: LiveProgress + NaNGuard + CheckpointZip + TiedMoEGrad")

Callbacks ready: LiveProgress + NaNGuard + CheckpointZip + TiedMoEGrad


In [6]:
# ============================================================
# 6. LOAD DATA — 9 per-category JSONL files
# ============================================================
data_dir = None
for cand in DATA_DIR_CANDIDATES:
    if cand and os.path.isdir(cand):
        if any(os.path.exists(os.path.join(cand, f)) for f in CATEGORY_FILES):
            data_dir = cand
            break

if data_dir is None:
    raise FileNotFoundError(
        "all_categorical_splits/ directory not found.\n"
        "Upload the 9 JSONL files as a Kaggle dataset and add it as input.\n"
        f"Searched: {DATA_DIR_CANDIDATES}"
    )

print(f"Found data directory: {data_dir}\n")

all_records = []
for fname in CATEGORY_FILES:
    fpath = os.path.join(data_dir, fname)
    if not os.path.exists(fpath):
        print(f"  [skip] {fname} not present")
        continue
    n = 0
    with open(fpath) as f:
        for line in f:
            if not line.strip():
                continue
            rec = json.loads(line)
            if "category" not in rec or not rec["category"]:
                rec["category"] = fname.replace("train_cot_", "").replace(".jsonl", "")
            all_records.append(rec)
            n += 1
    print(f"  loaded {n:>5} from {fname}")

print(f"\nTOTAL records loaded: {len(all_records)}")

Found data directory: /kaggle/input/datasets/asharamkanderiwal/nvidia-dataset/all_categorical_splits

  loaded  2728 from train_cot_bit_manipulation.jsonl
  loaded  1576 from train_cot_cipher.jsonl
  loaded   659 from train_cot_cryptarithm_deduce.jsonl
  loaded   164 from train_cot_cryptarithm_guess.jsonl
  loaded   540 from train_cot_equation_numeric_deduce.jsonl
  loaded   111 from train_cot_equation_numeric_guess.jsonl
  loaded  1597 from train_cot_gravity.jsonl
  loaded  1576 from train_cot_numeral.jsonl
  loaded  1594 from train_cot_unit_conversion.jsonl

TOTAL records loaded: 10545


In [7]:
# ============================================================
# 7. TOKENIZE & FORMAT — apply chat template, keep category labels
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

all_texts  = []
all_labels = []
fallback_template = 0

for rec in all_records:
    msgs = [m for m in rec["messages"] if m["role"] != "system"]
    try:
        text = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=False
        )
    except Exception:
        fallback_template += 1
        text = (
            f"<|im_start|>user\n{msgs[0]['content']}<|im_end|>\n"
            f"<|im_start|>assistant\n{msgs[-1]['content']}<|im_end|>"
        )
    all_texts.append(text)
    all_labels.append(rec["category"])

if fallback_template:
    print(f"[warn] Used fallback Chat-ML template for {fallback_template} records")

hf_dataset = Dataset.from_dict({"text": all_texts, "label": all_labels})
print(f"\nFormatted dataset: {len(hf_dataset)} examples\n")

from collections import Counter
dist = Counter(all_labels)
print("Category distribution:")
for name, n in dist.most_common():
    print(f"  {name:30s} {n:5d}  ({100*n/len(all_labels):.1f}%)")

print("\nSample (first 400 chars):")
print(hf_dataset[0]['text'][:400])


Formatted dataset: 10545 examples

Category distribution:
  bit_manipulation                2728  (25.9%)
  gravity                         1597  (15.1%)
  unit_conversion                 1594  (15.1%)
  cipher                          1576  (14.9%)
  numeral                         1576  (14.9%)
  cryptarithm_deduce               659  (6.2%)
  equation_numeric_deduce          540  (5.1%)
  cryptarithm_guess                164  (1.6%)
  equation_numeric_guess           111  (1.1%)

Sample (first 400 chars):
<|im_start|>system
<|im_end|>
<|im_start|>user
In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. The transformation involves operations like bit shifts, rotations, XOR, AND, OR, NOT, and possibly majority or choice functions.

Here are some examples of input -> output:
01010001 -> 11011101
00001001 -> 01101101
00010101 -> 01010101
11111111 -> 10000001
10011101 


In [8]:
# ============================================================
# 8. TOKEN LENGTH DIAGNOSTIC + DROP OVERSIZED SAMPLES
# ============================================================
# Print per-category length distribution BEFORE filtering so we see
# exactly how many samples are oversized per category.

print(f"Counting tokens for {len(hf_dataset)} samples...")

def get_token_length(example):
    ids = tokenizer(example['text'], truncation=False,
                    return_attention_mask=False)['input_ids']
    return {'token_len': len(ids)}

hf_dataset = hf_dataset.map(get_token_length, desc="Counting tokens")

# Per-category length distribution
from collections import defaultdict
import statistics

cat_lens = defaultdict(list)
for ex in hf_dataset:
    cat_lens[ex['label']].append(ex['token_len'])

print(f"\nPer-category length stats (max_seq_len cutoff = {MAX_SEQ_LEN}):")
print(f"  {'Category':<28} {'count':>6} {'med':>5} {'p90':>5} {'p99':>5} {'max':>5} {'>cut':>6} {'%kept':>6}")
print(f"  {'-'*28} {'------':>6} {'-----':>5} {'-----':>5} {'-----':>5} {'-----':>5} {'------':>6} {'------':>6}")

total_kept = 0
total_all  = 0
for cat in sorted(cat_lens):
    lens = sorted(cat_lens[cat])
    n = len(lens)
    med = lens[n // 2]
    p90 = lens[int(n * 0.90)]
    p99 = lens[min(int(n * 0.99), n - 1)]
    mx  = lens[-1]
    over = sum(1 for l in lens if l > MAX_SEQ_LEN)
    kept = n - over
    pct  = 100 * kept / n
    total_kept += kept
    total_all  += n
    print(f"  {cat:<28} {n:>6} {med:>5} {p90:>5} {p99:>5} {mx:>5} {over:>6} {pct:>5.1f}%")

print(f"  {'-'*28} {'------':>6}")
print(f"  {'TOTAL':<28} {total_all:>6} kept={total_kept} ({100*total_kept/total_all:.1f}%)  dropped={total_all-total_kept}")

# Suggest a better MAX_SEQ_LEN if drop rate >15%
all_lens = sorted([l for lens in cat_lens.values() for l in lens])
suggestions = []
for target_pct in [85, 90, 95, 99]:
    idx = int(len(all_lens) * target_pct / 100) - 1
    suggestions.append((target_pct, all_lens[max(0, idx)]))

print(f"\nSuggested MAX_SEQ_LEN to keep:")
for pct, sl in suggestions:
    fits_msg = "(fits 96GB w/ batch=2)" if sl <= 6144 else "(needs batch=1 on 96GB)"
    print(f"  {pct}% of data → MAX_SEQ_LEN >= {sl}  {fits_msg}")

# ============================================================
# Strategy: TRUNCATE instead of DROP
# ============================================================
# Why: bit_manipulation samples are all 7000-8000 tokens. Dropping >MAX_SEQ_LEN
# would remove the ENTIRE bit_manipulation category (2728 samples / 26% of data).
# That breaks training — the model never sees that domain.
#
# Better: keep ALL samples; for oversized ones, take the LAST MAX_SEQ_LEN tokens.
# Why the END: in chain-of-thought puzzles, the answer/conclusion is at the END.
# Keeping the tail preserves the most important learning signal.
# The user prompt at the start gets cut, but the model still sees:
#   - middle of reasoning (gives context)
#   - full conclusion + answer (the actual learning target)

print(f"\nTruncate-strategy: keeping ALL samples; oversized ones tail-truncated to {MAX_SEQ_LEN}")
before = len(hf_dataset)
oversized = sum(1 for tl in hf_dataset['token_len'] if tl > MAX_SEQ_LEN)

def _tail_truncate(example):
    if example['token_len'] <= MAX_SEQ_LEN:
        return example
    # Tokenize -> tail-truncate -> detokenize
    ids = tokenizer(example['text'], truncation=False, return_attention_mask=False)['input_ids']
    tail_ids = ids[-MAX_SEQ_LEN:]
    example['text'] = tokenizer.decode(tail_ids, skip_special_tokens=False)
    return example

if oversized > 0:
    hf_dataset = hf_dataset.map(_tail_truncate, desc=f"Tail-truncating {oversized} oversized samples")
    print(f"  Tail-truncated {oversized} samples to last {MAX_SEQ_LEN} tokens (preserves answer)")
hf_dataset = hf_dataset.remove_columns(['token_len'])
print(f"  Kept {len(hf_dataset)} / {before}  (0 dropped)")

steps_estimate = len(hf_dataset) // (BATCH_SIZE * GRAD_ACCUM) * NUM_EPOCHS
print(f"\nEstimated optimizer steps : {steps_estimate}")
sec_per_step = 4 if not USE_QLORA else 7   # bf16 faster than QLoRA
print(f"Estimated time            : ~{steps_estimate * sec_per_step / 3600:.1f}–{steps_estimate * sec_per_step * 1.8 / 3600:.1f} hrs")


Counting tokens for 10545 samples...


Counting tokens:   0%|          | 0/10545 [00:00<?, ? examples/s]


Per-category length stats (max_seq_len cutoff = 6144):
  Category                      count   med   p90   p99   max   >cut  %kept
  ---------------------------- ------ ----- ----- ----- ----- ------ ------
  bit_manipulation               2728  7042  7535  7743  8040   2728   0.0%
  cipher                         1576  3189  4717  6251  7261     17  98.9%
  cryptarithm_deduce              659   266   279   287   291      0 100.0%
  cryptarithm_guess               164   267   278   285   287      0 100.0%
  equation_numeric_deduce         540  5923  6322  6692  7081    117  78.3%
  equation_numeric_guess          111   164   180   184   184      0 100.0%
  gravity                        1597  3688  4935  5758  6365      3  99.8%
  numeral                        1576   251   290   314   330      0 100.0%
  unit_conversion                1594  2536  3557  4283  4805      0 100.0%
  ---------------------------- ------
  TOTAL                         10545 kept=7680 (72.8%)  dropped=2865


Tail-truncating 2865 oversized samples:   0%|          | 0/10545 [00:00<?, ? examples/s]

  Tail-truncated 2865 samples to last 6144 tokens (preserves answer)
  Kept 10545 / 10545  (0 dropped)

Estimated optimizer steps : 5272
Estimated time            : ~5.9–10.5 hrs


In [9]:
# ============================================================
# 9. LOAD MODEL — bf16 + Mamba FAST PATH ENABLED
# ============================================================
# THE SINGLE BIGGEST CHANGE FROM v76:
#   v76 disabled the Mamba fast path because we lacked the Blackwell
#   sm_120 wheels (causal_conv1d + mamba_ssm). Pure Python scan stored
#   ~30 GB of intermediate tensors during backward → forced batch=1, seq=4096.
#
#   v77 installs the Blackwell wheels in Cell 1, then ENABLES the fast path
#   here. The fused CUDA kernel stores nothing during forward and uses a
#   custom backward that recomputes — net memory: ~3 GB instead of ~30 GB.

flash_whl = "/kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/flash_attn-2.8.3+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"
if os.path.exists(flash_whl):
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-index", flash_whl])
        print("[ok] flash_attn installed")
    except Exception as e:
        print(f"[warn] flash_attn install skipped: {e}")


torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

t_load = time.time()

if MODE == "int8":
    print("Loading base model in 8-bit (LLM.int8) — should take 5-10 min...")
    bnb_config = BitsAndBytesConfig(
        load_in_8bit=True,
        llm_int8_threshold=6.0,
        llm_int8_has_fp16_weight=False,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    model = prepare_model_for_kbit_training(
        model, use_gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
    )

elif MODE == "nf4":
    print("Loading base model in 4-bit NF4...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_use_double_quant=False,
        bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH, quantization_config=bnb_config,
        device_map="auto", trust_remote_code=True,
    )
    model = prepare_model_for_kbit_training(
        model, use_gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
    )

else:  # bf16 (DEFAULT for v77)
    print("Loading base model in bf16 (~3 minutes)...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        device_map={"": 0},
        trust_remote_code=True,
        dtype=torch.bfloat16,
        low_cpu_mem_usage=True,
        attn_implementation="eager",   # 0.85's choice — most stable for hybrid
    )
    model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False}
    )

# ============================================================
# CRITICAL: Enable Mamba CUDA fast path (the v76 → v77 game-changer)
# ============================================================
# v76 set this to False because wheels were missing → 30GB Python scan tensors.
# v77 installs Blackwell wheels in Cell 1, enables fast path here.

nemotron_mod = None
for _name, _m in list(sys.modules.items()):
    if "modeling_nemotron_h" in _name and hasattr(_m, "is_fast_path_available"):
        nemotron_mod = _m
        break

if nemotron_mod is None:
    print("[warn] modeling_nemotron_h module not found in sys.modules yet")
    # Trigger import via a forward-ready check
    for _name in list(sys.modules.keys()):
        if "nemotron_h" in _name:
            nemotron_mod = sys.modules[_name]
            break

if nemotron_mod is not None:
    print(f"[info] is_fast_path_available was: {nemotron_mod.is_fast_path_available}")
    if USE_MAMBA_FAST_PATH:
        # Verify CUDA kernels actually work before enabling
        try:
            from causal_conv1d import causal_conv1d_fn
            _x = torch.randn(1, 256, 32, device="cuda", dtype=torch.bfloat16)
            _w = torch.randn(256, 4, device="cuda", dtype=torch.bfloat16)
            causal_conv1d_fn(_x, _w, None, activation="silu")
            print("[ok] causal_conv1d CUDA kernel verified working")

            import mamba_ssm
            print(f"[ok] mamba_ssm v{mamba_ssm.__version__} loaded")

            nemotron_mod.is_fast_path_available = True
            print(f"[OK] Mamba FAST PATH ENABLED ✓ (~30GB memory recovered, ~5x speedup)")
        except Exception as e:
            print(f"[FAIL] Fast path kernel check failed: {e}")
            print("       Falling back to pure-Python scan (HIGH memory)")
            nemotron_mod.is_fast_path_available = False
    else:
        nemotron_mod.is_fast_path_available = False
        print("[info] Mamba fast path DISABLED (USE_MAMBA_FAST_PATH=False)")
else:
    print("[error] Could not locate modeling_nemotron_h module — fast path config skipped")

# ============================================================
# DTYPE SAFETY PATCH for quantized MoE — only needed for int8/nf4
# ============================================================
def _find_moe_class(model):
    seen = set()
    for module in model.modules():
        cls = type(module)
        if cls in seen:
            continue
        seen.add(cls)
        moe_method = getattr(cls, "moe", None)
        if callable(moe_method) and not isinstance(moe_method, type):
            return cls
    return None

if MODE in ("int8", "nf4"):
    import inspect, textwrap
    moe_cls = _find_moe_class(model)
    if moe_cls is None:
        print("[warn] No MoE class found")
    else:
        try:
            src = textwrap.dedent(inspect.getsource(moe_cls.moe))
            patches = [
                ("expert_output = expert(expert_input)",
                 "expert_output = expert(expert_input).to(final_hidden_states.dtype)"),
            ]
            applied = []
            for old, new in patches:
                if old in src and new not in src:
                    src = src.replace(old, new)
                    applied.append(old)
            if applied:
                exec_ns = {}
                exec(src, moe_cls.moe.__globals__, exec_ns)
                moe_cls.moe = exec_ns["moe"]
                print(f"[ok] Patched {moe_cls.__name__}.moe ({len(applied)} repl)")
        except Exception as e:
            print(f"[warn] MoE patch failed: {e}")

load_min = (time.time() - t_load) / 60
vram_gb = torch.cuda.memory_allocated() / 1e9
total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"\nModel loaded in {load_min:.1f} min  |  Mode: {MODE}")
print(f"VRAM allocated: {vram_gb:.2f} / {total_gb:.1f} GB  ({100*vram_gb/total_gb:.0f}%)")
print(f"VRAM headroom : {total_gb - vram_gb:.2f} GB")
print(f"Fast path     : {'ENABLED ✓' if (nemotron_mod and nemotron_mod.is_fast_path_available) else 'DISABLED ✗'}")


[ok] flash_attn installed
Loading base model in bf16 (~3 minutes)...


Loading weights:   0%|          | 0/6243 [00:00<?, ?it/s]

[info] is_fast_path_available was: True
[ok] causal_conv1d CUDA kernel verified working
[ok] mamba_ssm v2.3.1 loaded
[OK] Mamba FAST PATH ENABLED ✓ (~30GB memory recovered, ~5x speedup)

Model loaded in 7.4 min  |  Mode: bf16
VRAM allocated: 63.17 / 102.0 GB  (62%)
VRAM headroom : 38.80 GB
Fast path     : ENABLED ✓


In [10]:
# ============================================================
# 10. APPLY LoRA — Attention + Mamba + (TIED) MoE  + fp32 LoRA cast
# ============================================================
# Strategy decisions (each one a deliberate choice over 0.85's recipe):
#
#   1) ATTENTION: q/k/v/o adapted (same as 0.85)
#   2) MAMBA: in_proj, out_proj adapted (same as 0.85, in_proj has z-gate)
#   3) MoE EXPERTS: up_proj + down_proj adapted with WEIGHT TYING
#      - 0.85's trick: all 128 experts share one LoRA factor (broadcast)
#      - Why: routing is sparse, so per-expert LoRA wastes 99% of capacity
#      - Tying side: A on up_proj (input-side), B on down_proj (output-side)
#      - Tinker convention: A and B that touch hidden_dim are tied
#   4) lm_head: NOT adapted (we keep eval-server contract clean)
#
# NUMERICAL HYGIENE (matches 0.85, prevents NaN poisoning):
#   - LoRA params:    fp32  (high precision for tiny gradients)
#   - Base model:     bf16  (storage)
#   - MoE router:     fp32  (Nemotron-H requires it; routing softmax stability)

from collections import Counter
import torch.nn as nn

linear_suffixes = Counter()
for name, module in model.named_modules():
    if isinstance(module, nn.Linear):
        suffix = name.split(".")[-1]
        linear_suffixes[suffix] += 1

print("ALL Linear module suffixes in this model:")
for suffix, n in sorted(linear_suffixes.items(), key=lambda x: -x[1]):
    print(f"  {suffix:30s} {n:4d}")
print()

ATTENTION_NAMES = ["q_proj", "k_proj", "v_proj", "o_proj", "Wqkv", "qkv_proj"]
MAMBA_NAMES     = ["in_proj", "out_proj", "x_proj", "dt_proj"]
MLP_GATE_NAMES  = ["gate_proj", "gate_up_proj", "w1", "linear_fc1", "fc1"]
MLP_UP_NAMES    = ["up_proj", "w3", "wi_1"]
MLP_DOWN_NAMES  = ["down_proj", "w2", "linear_fc2", "fc2", "wo"]
EXCLUDE         = {"lm_head", "embed_tokens", "shared", "router",
                   "score", "classifier"}

LORA_TARGET_MODULES = []
seen = set()

def add_if_present(names, label):
    added = []
    for n in names:
        if n in linear_suffixes and n not in seen and n not in EXCLUDE:
            LORA_TARGET_MODULES.append(n)
            seen.add(n)
            added.append(n)
    if added:
        print(f"  {label:18s}: {added}")
    return added

print("Target module selection:")
attn_added  = add_if_present(ATTENTION_NAMES, "attention")
mamba_added = add_if_present(MAMBA_NAMES,     "mamba")

# MoE handling depends on MOE_LORA_MODE (set in Cell 4)
if MOE_LORA_MODE in ("tied", "full"):
    gate_added = add_if_present(MLP_GATE_NAMES, "moe gate")
    up_added   = add_if_present(MLP_UP_NAMES,   "moe up")
    down_added = add_if_present(MLP_DOWN_NAMES, "moe down")
    if MOE_LORA_MODE == "tied":
        print(f"  {'MoE strategy':<18s}: TIED weights across 128 experts (memory-frugal)")
    else:
        print(f"  {'MoE strategy':<18s}: FULL per-expert LoRA (high memory)")
else:
    print(f"  {'mlp / MoE':<18s}: SKIPPED (MOE_LORA_MODE='{MOE_LORA_MODE}')")
print()

print(f"Final LoRA target modules: {LORA_TARGET_MODULES}")

assert LORA_TARGET_MODULES, (
    f"No LoRA target modules detected!\nAll: {dict(linear_suffixes)}"
)

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.enable_input_require_grads()

# ============================================================
# fp32 LoRA + bf16 base + fp32 MoE router (numerical hygiene)
# ============================================================
# 0.85 does this. We add an explicit verification.
print("\nCasting LoRA params -> fp32, verifying base/router dtypes...")
n_lora_fp32 = 0
n_router_fp32 = 0
n_base_bf16 = 0
n_other = 0
for name, param in model.named_parameters():
    if ".lora_" in name:
        param.data = param.data.to(torch.float32)
        n_lora_fp32 += 1
    elif ".mixer.gate." in name:
        # NemotronH router weight + e_score_correction_bias are fp32 by design
        if param.dtype == torch.float32:
            n_router_fp32 += 1
        else:
            print(f"  [warn] router param not fp32: {name} -> {param.dtype}")
    else:
        if param.dtype == torch.bfloat16:
            n_base_bf16 += 1
        else:
            n_other += 1
print(f"  LoRA params (fp32)   : {n_lora_fp32}")
print(f"  Router params (fp32) : {n_router_fp32}")
print(f"  Base params (bf16)   : {n_base_bf16}")
print(f"  Other dtypes         : {n_other}  (should be 0 in bf16 mode)")

# ============================================================
# MoE WEIGHT TYING — keeps all 128 expert LoRAs identical
# ============================================================
# Conceptual setup:
#   - Each MoE expert has its own up_proj/down_proj LoRA (PEFT default)
#   - We MAKE THEM IDENTICAL by averaging/broadcasting per training step
#   - Adam stays in sync because gradients are summed, not averaged
#
# Why TIE A on up_proj and B on down_proj?
#   - up_proj.lora_A:  [r, hidden_dim]   — input projection
#   - up_proj.lora_B:  [intermediate, r] — per-expert (NOT tied)
#   - down_proj.lora_A:[r, intermediate] — per-expert (NOT tied)
#   - down_proj.lora_B:[hidden_dim, r]   — output projection
#   The "shared" side touches hidden_dim; tying it keeps expert specialization
#   in the intermediate dim while sharing the hidden-dim adaptation.

moe_tied_params = []
if MOE_LORA_MODE == "tied":
    w1_proj_names = ("gate_up_proj", "up_proj", "gate_proj", ".w1.")
    w2_proj_names = ("down_proj", ".w2.")
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if ".experts." not in name or ".lora_" not in name:
            continue
        is_w1 = any(p in name for p in w1_proj_names)
        is_w2 = any(p in name for p in w2_proj_names)
        is_A = ".lora_A." in name
        is_B = ".lora_B." in name
        should_tie = (is_w1 and is_A) or (is_w2 and is_B)
        if not should_tie:
            continue
        if param.dim() < 2 or param.shape[0] <= 1:
            continue
        moe_tied_params.append(param)

    def _tie_param_init():
        """Make all expert slices identical at start (mean-and-broadcast)."""
        with torch.no_grad():
            for p in moe_tied_params:
                mean = p.data.mean(dim=0, keepdim=True)
                p.data.copy_(mean.expand_as(p.data))

    def _tie_grads():
        """Sum gradients across expert dim, broadcast back. Called pre-step."""
        with torch.no_grad():
            for p in moe_tied_params:
                if p.grad is None:
                    continue
                grad_sum = p.grad.sum(dim=0, keepdim=True)
                p.grad.copy_(grad_sum.expand_as(p.grad))

    print(f"\n[MoE TYING] Identified {len(moe_tied_params)} params to tie")
    if moe_tied_params:
        print(f"  example shapes: {[tuple(p.shape) for p in moe_tied_params[:3]]}")
    _tie_param_init()
    print("  Initial state: all expert slices set to per-param mean (TIED)")
else:
    def _tie_grads():
        pass

# ============================================================
# Trainable parameter audit
# ============================================================
model.print_trainable_parameters()
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable params: {trainable/1e6:.1f}M")

if MOE_LORA_MODE == "tied":
    # Effective trainable = per-expert tied slices count as 1
    expert_param_count = sum(p[0:1].numel() for p in moe_tied_params)
    raw_expert_param_count = sum(p.numel() for p in moe_tied_params)
    effective = trainable - (raw_expert_param_count - expert_param_count)
    print(f"  Effective (tied)   : {effective/1e6:.1f}M  (saves {(raw_expert_param_count - expert_param_count)/1e6:.1f}M from tying)")

if trainable > 900e6:
    print(f"  [warn] >900M trainable — likely high memory; consider MOE_LORA_MODE='exclude' or 'tied'")
elif trainable < 5e6:
    print("  [warn] <5M trainable — too few targets matched")
else:
    print(f"  [ok] healthy LoRA size")

vram_after_lora = torch.cuda.memory_allocated() / 1e9
print(f"\nVRAM after LoRA: {vram_after_lora:.2f} GB")
print(f"VRAM headroom   : {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved())/1e9:.2f} GB")

try:
    import triton.backends.nvidia.compiler as nv_compiler
    os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = "/tmp/ptxas-blackwell"
    nv_compiler.get_ptxas_version = lambda arch: "12.0"
except Exception as e:
    print(f"[warn] Triton compiler fix skipped: {e}")


ALL Linear module suffixes in this model:
  up_proj                        2967
  down_proj                      2967
  in_proj                          23
  out_proj                         23
  q_proj                            6
  k_proj                            6
  v_proj                            6
  o_proj                            6
  lm_head                           1

Target module selection:
  attention         : ['q_proj', 'k_proj', 'v_proj', 'o_proj']
  mamba             : ['in_proj', 'out_proj']
  moe up            : ['up_proj']
  moe down          : ['down_proj']
  MoE strategy      : TIED weights across 128 experts (memory-frugal)

Final LoRA target modules: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'in_proj', 'out_proj', 'up_proj', 'down_proj']

Casting LoRA params -> fp32, verifying base/router dtypes...
  LoRA params (fp32)   : 12008
  Router params (fp32) : 23
  Base params (bf16)   : 6197
  Other dtypes         : 0  (should be 0 in bf16 mode)

[MoE TYING] Identi

In [13]:
# ============================================================
# 11. TRAINING — Stratified SFT  +  CCE loss  +  plain AdamW
# ============================================================
# All v77 fixes integrated:
#   - CCE forward patch          → ~17 GB saved (no logits materialization)
#   - torch.optim.AdamW          → Blackwell-stable (no UVM crash)
#   - max_grad_norm=1.0          → fp32 LoRA tolerates standard clip
#   - save_steps=200 + epoch zip → resume across sessions
#   - NaN guard + Tied-MoE callbacks → from Cell 6
#   - compute_loss override      → bypass TRL entropy_from_logits crash on None

from torch.utils.data import Sampler
import random as _random

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

EFFECTIVE_BATCH = BATCH_SIZE * GRAD_ACCUM

# ---- RESUME ACROSS SESSIONS ----
RESUME_FROM_CHECKPOINT = None

# ============================================================
# CCE FORWARD PATCH — no logits tensor ever materialized
# ============================================================
if USE_CCE:
    try:
        from cut_cross_entropy import linear_cross_entropy
        print("[ok] cut_cross_entropy imported")

        _base = model
        while hasattr(_base, "model"):
            _base = _base.model

        if not (hasattr(_base, "backbone") and hasattr(_base, "lm_head")):
            print(f"[warn] CCE patch skipped: model layout missing .backbone/.lm_head")
            print(f"       _base type: {type(_base).__name__}")
            USE_CCE = False
        else:
            _lm_head_module = _base.lm_head
            _orig_forward   = _base.forward

            def _cce_forward(input_ids=None, attention_mask=None, labels=None, **kwargs):
                backbone_out = _base.backbone(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    **{k: v for k, v in kwargs.items()
                       if k in ("position_ids", "past_key_values", "use_cache",
                                "inputs_embeds", "cache_position")},
                )
                hidden_states = backbone_out[0]

                if hasattr(_lm_head_module, "base_layer"):
                    base_w = _lm_head_module.base_layer.weight
                    if hasattr(_lm_head_module, "lora_A") and "default" in _lm_head_module.lora_A:
                        lora_A = _lm_head_module.lora_A["default"].weight
                        lora_B = _lm_head_module.lora_B["default"].weight
                        scaling = _lm_head_module.scaling["default"]
                        lm_weight = base_w + scaling * (lora_B @ lora_A)
                    else:
                        lm_weight = base_w
                else:
                    lm_weight = _lm_head_module.weight

                if labels is not None:
                    shift_hidden = hidden_states[..., :-1, :].contiguous()
                    shift_labels = labels[..., 1:].contiguous()
                    valid = shift_labels != -100
                    if valid.any():
                        loss = linear_cross_entropy(
                            shift_hidden,
                            lm_weight,
                            shift_labels.masked_fill(~valid, 0),
                            reduction="none",
                        )
                        loss = (loss * valid.float()).sum() / valid.float().sum().clamp(min=1)
                    else:
                        loss = shift_hidden.sum() * 0.0
                else:
                    loss = None

                from transformers.modeling_outputs import CausalLMOutputWithPast
                return CausalLMOutputWithPast(
                    loss=loss,
                    logits=None,  # TRL's compute_loss is overridden below to handle this
                    past_key_values=getattr(backbone_out, "past_key_values", None),
                    hidden_states=getattr(backbone_out, "hidden_states", None),
                    attentions=getattr(backbone_out, "attentions", None),
                )

            _base.forward = _cce_forward
            print("[OK] CCE forward patched ✓ (~17GB saved, no logits materialization)")
    except ImportError as e:
        print(f"[warn] CCE not available: {e}")
        USE_CCE = False
    except Exception as e:
        print(f"[warn] CCE patch failed: {e}")
        import traceback; traceback.print_exc()
        USE_CCE = False
else:
    print("[info] CCE disabled — using standard CrossEntropy (uses ~17GB more VRAM)")

# ============================================================
# Stratified sampler — coherent gradient per category (OUR EDGE)
# ============================================================
def build_stratified_index_order(labels, chunk_size, seed=0):
    buckets = {}
    for i, lbl in enumerate(labels):
        buckets.setdefault(lbl, []).append(i)
    rng = _random.Random(seed)
    for lbl in buckets:
        rng.shuffle(buckets[lbl])
    order = []
    active = list(buckets.keys())
    rng.shuffle(active)
    while active:
        next_active = []
        for lbl in active:
            take = buckets[lbl][:chunk_size]
            buckets[lbl] = buckets[lbl][chunk_size:]
            order.extend(take)
            if buckets[lbl]:
                next_active.append(lbl)
        active = next_active
    return order


class PrecomputedOrderSampler(Sampler):
    def __init__(self, labels, chunk_size, num_epochs, base_seed=1337):
        self.labels      = list(labels)
        self.chunk_size  = chunk_size
        self.num_epochs  = num_epochs
        self.base_seed   = base_seed
        self.epoch       = 0
        self._current_order = build_stratified_index_order(
            self.labels, chunk_size, seed=base_seed
        )
    def set_epoch(self, epoch):
        self.epoch = epoch
        self._current_order = build_stratified_index_order(
            self.labels, self.chunk_size, seed=self.base_seed + epoch
        )
    def __iter__(self):
        return iter(self._current_order)
    def __len__(self):
        return len(self.labels)


# ============================================================
# CRITICAL FIX: compute_loss override — bypass TRL entropy_from_logits(None) crash
# ============================================================
# TRL >=0.11 unconditionally calls entropy_from_logits(outputs.logits) inside
# compute_loss when liger_kernel is off. Our CCE returns logits=None to save
# 17 GB → that path crashes with "NoneType has no attribute 'shape'".
#
# Fix: override compute_loss to:
#   1) Run the model (CCE patch already returned loss)
#   2) Return outputs.loss directly without touching outputs.logits

def _cce_compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
    """Compute loss without touching outputs.logits (which is None when CCE is on)."""
    outputs = model(**inputs)
    if outputs.loss is not None:
        loss = outputs.loss
    else:
        loss = torch.zeros((), device=next(model.parameters()).device, requires_grad=True)
    return (loss, outputs) if return_outputs else loss


class StratifiedSFTTrainer(SFTTrainer):
    def __init__(self, *args, stratified_labels=None, chunk_size=16,
                 num_epochs=3, **kwargs):
        self._strat_labels = stratified_labels
        self._strat_chunk  = chunk_size
        self._strat_epochs = num_epochs
        super().__init__(*args, **kwargs)

    def _get_train_sampler(self, *args, **kwargs):
        if self._strat_labels is None:
            return super()._get_train_sampler(*args, **kwargs)
        return PrecomputedOrderSampler(
            labels=self._strat_labels,
            chunk_size=self._strat_chunk,
            num_epochs=self._strat_epochs,
        )

    # CCE-safe loss computation (skips TRL's entropy_from_logits)
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        return _cce_compute_loss(self, model, inputs, return_outputs, num_items_in_batch)

    # Plain torch.optim.AdamW (Blackwell-stable, no paged_adamw_8bit UVM crash)
    def create_optimizer(self):
        if self.optimizer is None:
            decay_params = [p for p in self.model.parameters() if p.requires_grad]
            self.optimizer = torch.optim.AdamW(
                decay_params,
                lr=self.args.learning_rate,
                betas=(0.9, 0.95),
                eps=1e-8,
                weight_decay=0.0,
            )
            print(f"[ok] Optimizer: torch.optim.AdamW on {sum(p.numel() for p in decay_params)/1e6:.1f}M params (Blackwell-stable)")
        return self.optimizer


class _PlainTrainer(SFTTrainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        return _cce_compute_loss(self, model, inputs, return_outputs, num_items_in_batch)
    def create_optimizer(self):
        if self.optimizer is None:
            decay_params = [p for p in self.model.parameters() if p.requires_grad]
            self.optimizer = torch.optim.AdamW(
                decay_params, lr=self.args.learning_rate,
                betas=(0.9, 0.95), eps=1e-8, weight_decay=0.0,
            )
        return self.optimizer


# ============================================================
# SFTConfig — bf16, plain AdamW, step + epoch checkpoints
# ============================================================
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LR,
    logging_steps=10,
    bf16=True,
    max_grad_norm=1.0,
    optim="adamw_torch",
    lr_scheduler_type="cosine",
    warmup_steps=WARMUP_STEPS,
    save_strategy="steps",
    save_steps=SAVE_EVERY_N_STEPS,
    save_total_limit=1,             # was 3 — disk-full crash on /kaggle/working
    save_only_model=True,           # don't save optimizer state (~5 GB each save)
    report_to="wandb" if WANDB_AVAILABLE else "none",
    run_name=WANDB_RUN_NAME if WANDB_AVAILABLE else None,
    dataset_text_field="text",
    max_length=MAX_SEQ_LEN,
    packing=USE_PACKING,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    dataloader_prefetch_factor=2,
    remove_unused_columns=False,
)

labels_for_sampler = list(hf_dataset['label']) if USE_STRATIFIED_BATCHING else None

callbacks = [LiveProgressCallback(), nan_callback, ckpt_callback]
if MOE_LORA_MODE == "tied":
    callbacks.append(tied_callback)
    print("[info] TiedMoEGrad callback active (gradients summed across experts)")

if USE_STRATIFIED_BATCHING:
    print(f"Using STRATIFIED batching (chunk = {EFFECTIVE_BATCH} samples/category)")
    trainer = StratifiedSFTTrainer(
        model=model,
        train_dataset=hf_dataset,
        processing_class=tokenizer,
        args=training_args,
        callbacks=callbacks,
        stratified_labels=labels_for_sampler,
        chunk_size=EFFECTIVE_BATCH,
        num_epochs=NUM_EPOCHS,
    )
else:
    print("Using standard RANDOM batching")
    trainer = _PlainTrainer(
        model=model,
        train_dataset=hf_dataset,
        processing_class=tokenizer,
        args=training_args,
        callbacks=callbacks,
    )

steps_per_epoch = trainer.state.max_steps // NUM_EPOCHS if hasattr(trainer.state, 'max_steps') else "?"
print("=" * 60)
print(f"  v7.7 Training — surpass-0.85 stack")
print("=" * 60)
print(f"  Samples       : {len(hf_dataset)}  |  epochs={NUM_EPOCHS}  |  max_seq={MAX_SEQ_LEN}")
print(f"  Batch         : {BATCH_SIZE}×{GRAD_ACCUM} = {EFFECTIVE_BATCH} eff   steps/epoch≈{steps_per_epoch}")
print(f"  LR            : {LR:.1e}  warmup={WARMUP_STEPS}  schedule=cosine")
print(f"  LoRA          : r={LORA_RANK}, α={LORA_ALPHA}, MoE={MOE_LORA_MODE}")
print(f"  Loss kernel   : {'CUT-CROSS-ENTROPY ✓' if USE_CCE else 'standard CrossEntropy'}")
print(f"  compute_loss  : OVERRIDDEN (bypasses TRL entropy_from_logits crash on None)")
print(f"  Mamba kernel  : {'FAST PATH ✓' if (nemotron_mod and nemotron_mod.is_fast_path_available) else 'PYTHON SCAN ✗'}")
print(f"  Optimizer     : torch.optim.AdamW (β=0.9,0.95)")
print(f"  Stratified    : {USE_STRATIFIED_BATCHING}")
print(f"  Checkpoints   : every {SAVE_EVERY_N_STEPS} steps + per-epoch zip")
print(f"  NaN guard     : ON (halts after 2 consecutive non-finite losses)")
print(f"  Resume from   : {RESUME_FROM_CHECKPOINT or '(fresh start)'}")
print(f"  W&B           : {'offline' if WANDB_AVAILABLE else 'disabled'}\n")

t0 = time.time()
trainer.train(resume_from_checkpoint=RESUME_FROM_CHECKPOINT)
elapsed_hrs = (time.time() - t0) / 3600
print(f"\nTraining complete! Time: {elapsed_hrs:.2f} hrs")

ckpt_callback.print_summary()

peak_vram = torch.cuda.max_memory_allocated() / 1e9
print(f"\nPeak VRAM during training: {peak_vram:.2f} GB / 95 GB")


[ok] cut_cross_entropy imported
[OK] CCE forward patched ✓ (~17GB saved, no logits materialization)
[info] TiedMoEGrad callback active (gradients summed across experts)
Using standard RANDOM batching


Adding EOS to train dataset:   0%|          | 0/10545 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/10545 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/10545 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 11, 'pad_token_id': 11}.


  v7.7 Training — surpass-0.85 stack
  Samples       : 10545  |  epochs=1  |  max_seq=6144
  Batch         : 2×1 = 2 eff   steps/epoch≈0
  LR            : 1.0e-04  warmup=50  schedule=cosine
  LoRA          : r=32, α=64, MoE=tied
  Loss kernel   : CUT-CROSS-ENTROPY ✓
  compute_loss  : OVERRIDDEN (bypasses TRL entropy_from_logits crash on None)
  Mamba kernel  : FAST PATH ✓
  Optimizer     : torch.optim.AdamW (β=0.9,0.95)
  Stratified    : False
  Checkpoints   : every 200 steps + per-epoch zip
  NaN guard     : ON (halts after 2 consecutive non-finite losses)
  Resume from   : (fresh start)
  W&B           : offline



Training:   0%|          | 0/3082 [00:00<?, ?step/s]

Step,Training Loss
10,0.455340
20,0.477675
30,0.374647
40,0.274587
50,0.236068
60,0.149789
70,0.108737
80,0.104588
90,0.093777
100,0.057365


wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run


RuntimeError: [enforce fail at inline_container.cc:668] . unexpected pos 5330345024 vs 5330344920

In [ ]:
# ============================================================
# 12. SAVE FINAL ADAPTER — plain LoRA  (vLLM compatible)
# ============================================================
trainer.model.save_pretrained(OUTPUT_DIR)

config_path = os.path.join(OUTPUT_DIR, "adapter_config.json")
with open(config_path) as f:
    adapter_config = json.load(f)

adapter_config["base_model_name_or_path"] = "metric/nemotron-3-nano-30b-a3b-bf16"
with open(config_path, "w") as f:
    json.dump(adapter_config, f, indent=2)

print(f"base_model_name_or_path -> {adapter_config['base_model_name_or_path']}")
print(f"peft_type              -> {adapter_config.get('peft_type')}")
print(f"r / alpha              -> {adapter_config.get('r')} / {adapter_config.get('lora_alpha')}")
print(f"use_dora               -> {adapter_config.get('use_dora', False)}  (must be False)")
print(f"use_rslora             -> {adapter_config.get('use_rslora', False)}  (must be False)")

try:
    from safetensors import safe_open
    with safe_open(os.path.join(OUTPUT_DIR, "adapter_model.safetensors"),
                   framework="pt") as f:
        keys  = list(f.keys())
        norms = [f.get_tensor(k).norm().item() for k in keys[:5]]
    print(f"\nAdapter tensors: {len(keys)} parameters")
    print(f"First 5 weight norms: {[f'{n:.4f}' for n in norms]}")
    if all(n < 0.001 for n in norms):
        print("WARNING: Norms near 0 — adapter may be untrained!")
    else:
        print("Adapter looks healthy (non-zero weights).")
except Exception as e:
    print(f"Could not verify safetensors: {e}")

print(f"\nFiles in {OUTPUT_DIR}:")
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    if os.path.isfile(fpath):
        size_mb = os.path.getsize(fpath) / 1024 / 1024
        print(f"  {fname}  ({size_mb:.2f} MB)")

In [ ]:
# ============================================================
# 13. ZIP FINAL ADAPTER + LIST PER-EPOCH CHECKPOINTS
# ============================================================
ZIP_PATH = "/kaggle/working/adapter.zip"
if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

file_count = 0
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in sorted(os.listdir(OUTPUT_DIR)):
        fpath = os.path.join(OUTPUT_DIR, fname)
        if os.path.isfile(fpath):
            zf.write(fpath, arcname=fname)
            file_count += 1

with zipfile.ZipFile(ZIP_PATH) as zf:
    contents = zf.namelist()

zip_mb = os.path.getsize(ZIP_PATH) / 1024 / 1024

print("=" * 60)
print(f"  FINAL adapter.zip  ({zip_mb:.1f} MB)  — {file_count} files")
print(f"  Contents: {contents}")
print("=" * 60)

ckpt_zips = sorted(
    [f for f in os.listdir(CKPT_DIR) if f.endswith(".zip")],
    key=lambda x: int(x.replace("adapter_epoch_", "").replace(".zip", ""))
    if x.replace("adapter_epoch_", "").replace(".zip", "").isdigit() else 0
)

print(f"\nPer-epoch checkpoints in {CKPT_DIR}:")
total_ckpt_mb = 0
for zname in ckpt_zips:
    zpath = os.path.join(CKPT_DIR, zname)
    mb = os.path.getsize(zpath) / 1024 / 1024
    total_ckpt_mb += mb
    epoch_num = zname.replace("adapter_epoch_", "").replace(".zip", "")
    loss = ckpt_callback.epoch_losses.get(int(epoch_num), float("nan")) if epoch_num.isdigit() else float("nan")
    print(f"  {zname:<30}  {mb:6.1f} MB  loss={loss:.4f}")

print(f"\nTotal checkpoint storage: {total_ckpt_mb:.1f} MB  ({len(ckpt_zips)} files)")
print("\nTip: Use the checkpoint with the LOWEST loss for your submission.")

assert "adapter_config.json" in contents, "MISSING adapter_config.json!"
assert "adapter_model.safetensors" in contents, "MISSING adapter_model.safetensors!"

In [ ]:
# ============================================================
# 14. FINAL VERIFICATION — eval-server compliance + memory report
# ============================================================
print("=" * 60)
print("  v7.6 TRAINING SUMMARY")
print("=" * 60)

with open(os.path.join(OUTPUT_DIR, "adapter_config.json")) as f:
    final_cfg = json.load(f)

mode_str = "Hybrid, 4-bit NF4 base (QLoRA)" if USE_QLORA else "Hybrid, bf16 base"
print(f"\n  Model             : Nemotron-3-Nano-30B-A3B ({mode_str})")
print(f"  base_model_name   : {final_cfg.get('base_model_name_or_path')}")
print(f"  LoRA rank (r)     : {final_cfg.get('r')}")
print(f"  LoRA alpha        : {final_cfg.get('lora_alpha')}")
print(f"  Target modules    : {final_cfg.get('target_modules')}")
print(f"  Dropout           : {final_cfg.get('lora_dropout')}")
print(f"  use_dora          : {final_cfg.get('use_dora', False)}")
print(f"  use_rslora        : {final_cfg.get('use_rslora', False)}")
print(f"\n  Dataset           : {len(hf_dataset)} examples (after filtering)")
print(f"  Categories        : {len(set(hf_dataset['label']))}")
print(f"  Epochs            : {NUM_EPOCHS}")
print(f"  Learning rate     : {LR}")
print(f"  Effective batch   : {BATCH_SIZE * GRAD_ACCUM}")
print(f"  Training time     : {elapsed_hrs:.2f} hrs")
print(f"  Peak VRAM         : {peak_vram:.2f} GB")
print(f"\n  Adapter zip       : {ZIP_PATH} ({zip_mb:.1f} MB)")
print(f"  Adapter files     : {contents}")

targets = final_cfg.get('target_modules', [])
has_mamba = 'in_proj' in targets       # Mamba's in_proj contains z-gate
has_attn  = 'q_proj' in targets or 'qkv_proj' in targets
has_mlp_gate = any(m in targets for m in
                   ["gate_proj", "gate_up_proj", "w1",
                    "linear_fc1", "fc1", "gate", "wi_0", "wi"])
checks = [
    ("base_model = metric/...",       final_cfg.get('base_model_name_or_path') == 'metric/nemotron-3-nano-30b-a3b-bf16'),
    ("peft_type = LORA",               final_cfg.get('peft_type') == 'LORA'),
    ("attention IN targets",           has_attn),
    ("mamba in_proj IN targets",       has_mamba),
    ("lm_head NOT in targets",         'lm_head' not in targets),
    ("embed_tokens NOT in targets",    'embed_tokens' not in targets),
    ("dropout = 0",                    final_cfg.get('lora_dropout', -1) == 0.0),
    ("rank = 32",                      final_cfg.get('r') == 32),
    ("alpha = 64",                     final_cfg.get('lora_alpha') == 64),
    ("use_dora = False/absent",        not final_cfg.get('use_dora', False)),
    ("use_rslora = False/absent",      not final_cfg.get('use_rslora', False)),
    ("modules_to_save empty/absent",   not final_cfg.get('modules_to_save')),
]

print(f"\n  Verification checks:")
all_ok = True
for name, passed in checks:
    status = "PASS" if passed else "FAIL"
    if not passed:
        all_ok = False
    print(f"    [{status}] {name}")

# Informational, not a FAIL: MLP gate is optional on Nemotron-H
gate_status = "PASS" if has_mlp_gate else "INFO (Mamba in_proj covers gating)"
print(f"    [{gate_status}] explicit MLP gate IN targets")

if all_ok:
    print(f"\n  All required checks passed! Adapter is eval-server compliant.")
    print(f"  -> Download adapter.zip (or best per-epoch checkpoint) from Kaggle Output")
    print(f"  -> Use with nemotron_v75_submission.ipynb (or any plain-LoRA submission)")
else:
    print(f"\n  WARNING: Some checks failed — review before submitting.")

if WANDB_AVAILABLE:
    wandb.log({
        "final/training_hours": elapsed_hrs,
        "final/dataset_size": len(hf_dataset),
        "final/adapter_zip_mb": zip_mb,
        "final/peak_vram_gb": peak_vram,
    })
    wandb.finish()
    wandb_dir = os.path.join(WANDB_DIR, "wandb")
    wandb_zip = "/kaggle/working/wandb_logs.zip"
    if os.path.exists(wandb_dir):
        with zipfile.ZipFile(wandb_zip, "w", zipfile.ZIP_DEFLATED) as zf:
            for root, dirs, files in os.walk(wandb_dir):
                for fname in files:
                    fpath = os.path.join(root, fname)
                    arcname = os.path.relpath(fpath, WANDB_DIR)
                    zf.write(fpath, arcname=arcname)
        wandb_zip_mb = os.path.getsize(wandb_zip) / 1024 / 1024
        print(f"\n  W&B logs zipped: {wandb_zip} ({wandb_zip_mb:.1f} MB)")
